In [1]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
import torch, subprocess, shutil
print("-" * 55)
print("PyTorch :", torch.__version__)
print("CUDA    :", torch.version.cuda)
print("GPU     :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (CPU only — slower)")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM    : {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
print("RAM     :", end=" ")
!free -h | awk '/Mem:/ {print $3 " used / " $2 " total"}'


name, memory.total [MiB], memory.free [MiB]
Tesla T4, 15360 MiB, 14913 MiB
-------------------------------------------------------
PyTorch : 2.11.0+cu128
CUDA    : 12.8
GPU     : Tesla T4
VRAM    : 15.5 GB free / 15.6 GB total
RAM     : 1.5Gi used / 12Gi total


In [2]:
%%capture
!pip install -q --upgrade pip
!pip install -q transformers accelerate bitsandbytes
!pip install -q faster-whisper gradio yt-dlp huggingface_hub

import warnings, os
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("✅ All dependencies installed")


In [3]:
# ============ ⚙️ SETTINGS ============
WHISPER_MODEL = "large-v3"   # tiny | base | small | medium | large-v3  (large-v3 = best for Hindi)
LLM_MODEL     = "Qwen/Qwen2.5-7B-Instruct"   # fits in 15GB in 4-bit
MAX_NEW_TOKENS = 2200
TEMPERATURE    = 0.75
# ====================================

print(f"Whisper : {WHISPER_MODEL}")
print(f"LLM     : {LLM_MODEL} (4-bit quantised)")


Whisper : large-v3
LLM     : Qwen/Qwen2.5-7B-Instruct (4-bit quantised)


In [4]:
from faster_whisper import WhisperModel
import torch, gc

print("Loading Whisper (first run downloads the model)…")
whisper = WhisperModel(
    WHISPER_MODEL,
    device="cuda" if torch.cuda.is_available() else "cpu",
    compute_type="float16" if torch.cuda.is_available() else "int8",
)
print("✅ Whisper ready\n")

def transcribe_audio(path):
    """Returns plain transcript text from an audio/video file."""
    segments, info = whisper.transcribe(
        path, beam_size=5, vad_filter=True,
        vad_parameters={"min_silence_duration_ms": 500},
    )
    text = " ".join(s.text.strip() for s in segments)
    print(f"   language={info.language}  prob={info.language_probability:.2f}  "
          f"duration={info.duration:.0f}s  chars={len(text)}")
    return text


Loading Whisper (first run downloads the model)…


✅ Whisper ready



In [5]:
import torch, gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

gc.collect(); torch.cuda.empty_cache()

quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {LLM_MODEL} … (first run downloads ~5 GB)")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    quantization_config=quant,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()
gc.collect(); torch.cuda.empty_cache()
print("✅ Local LLM ready — no API key, fully offline from here\n")


Loading Qwen/Qwen2.5-7B-Instruct … (first run downloads ~5 GB)


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (3 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Local LLM ready — no API key, fully offline from here



In [6]:
import json, re, textwrap

SYSTEM_PROMPT = r"""
You are a world-class YouTube Shorts SEO strategist and packaging expert.
You write viral metadata for short-form video. You think in terms of RETENTION
and CLICK-THROUGH, not keyword stuffing.

NON-NEGOTIABLE RULES:
1. Titles are MAX 60 characters. Count them. No hashtags in the title.
2. The FIRST 40 characters of a title must contain the main search keyword,
   because mobile truncates after that.
3. Lead with the DEMAND keyword (what people type), put the product/brand name
   SECOND. Never lead with an unknown brand name.
4. One language only. Title, thumbnail text and opening hook must match.
5. Description line 1 must contain the exact search keyword, in the first 60 chars.
6. 5 hashtags MAX. Broad topic tag first.
7. 25 tags. Mix: broad + long-tail + question-form + the language of the audience.
   Total under 480 characters. Comma separated.
8. Never promise view counts. Never say "this will go viral".
9. Be honest in the diagnosis: if the real problem is the hook or the topic
   (not the metadata), say so clearly and rank the problems by impact.
10. Thumbnail text is MAX 3 words.

OUTPUT FORMAT — return ONLY valid JSON, no markdown fences, with exactly these keys:
{
  "niche": "short phrase describing the video topic",
  "diagnosis": [
    {"problem": "...", "severity": "High|Medium|Low", "fix": "..."}
  ],
  "titles": ["title1", "title2", "title3", "title4", "title5"],
  "description": "the full description text with line breaks",
  "tags": ["tag1", "tag2", "...25 of them"],
  "hashtags": ["#A", "#B", "#C", "#D", "#E"],
  "thumbnail_prompt": "a detailed prompt for an AI image generator, vertical 9:16",
  "thumbnail_text": "3 words max",
  "hook_advice": "concrete advice for the first 2 seconds of the video",
  "retention_tips": ["tip1", "tip2", "tip3", "tip4"],
  "common_mistakes": ["mistake1", "mistake2", "mistake3"]
}
""".strip()


def build_prompt(transcript, language, niche, current_title, platform):
    lang_note = {
        "Hindi":  "Write the titles, thumbnail text and description in HINDI (Devanagari). Tags may mix Hindi + English.",
        "Hinglish":"Write titles and thumbnail text in HINGLISH (Roman script). Description in Hinglish. Tags mix Hindi + English.",
        "English":"Write everything in English.",
    }.get(language, "Hinglish")

    return f"""{SYSTEM_PROMPT}

--- TARGET SETTINGS ---
Language : {language} — {lang_note}
Niche    : {niche or "infer it from the transcript"}
Platform : {platform or "YouTube Shorts"}
Current title (if any): {current_title or "(none)"}

--- VIDEO TRANSCRIPT ---
{transcript[:9000]}

Now produce the complete JSON SEO package."""


def parse_json(text):
    """Robustly pull a JSON object out of model output."""
    t = text.strip()
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", t, flags=re.S)

    # Attempt 1: Direct parse
    try:
        return json.loads(t)
    except json.JSONDecodeError as e:
        if "Extra data" in str(e):
            try:
                # Try parsing only up to the error position
                return json.loads(t[:e.pos])
            except Exception:
                pass  # Fall through to other attempts
        pass # Fall through for other JSONDecodeError types

    # Attempt 2: Find first '{' and last '}'
    start = t.find("{")
    end = t.rfind("}")

    if start != -1 and end > start:
        candidate_json = t[start : end + 1]
        try:
            return json.loads(candidate_json)
        except json.JSONDecodeError as e:
            if "Extra data" in str(e):
                try:
                    # Try parsing only up to the error position
                    return json.loads(candidate_json[:e.pos])
                except Exception:
                    pass  # Fall through to last resort fix
            # Attempt 3 (originally "last resort"): fix trailing commas
            try:
                fixed = re.sub(r",\s*([}\]])", r"\1", candidate_json)
                return json.loads(fixed)
            except json.JSONDecodeError as e_fixed:
                # If 'Extra data', try parsing only up to the error position
                if "Extra data" in str(e_fixed):
                    temp_fixed = fixed[:e_fixed.pos] # Start with truncation at e.pos
                    # Try to iteratively truncate from the end until valid JSON is found
                    # or it becomes too short to be meaningful
                    for i in range(len(temp_fixed), max(len(temp_fixed) - 500, 0), -1): # Iterate backwards
                        try:
                            # Try to parse current truncated string
                            return json.loads(temp_fixed[:i])
                        except json.JSONDecodeError as inner_e:
                            # Continue if it's still a JSONDecodeError, but break if it's Extra data again (shouldn't happen with current logic)
                            if "Extra data" not in str(inner_e):
                                continue
                            else: # Extra data here implies an issue with this trimming strategy, stop
                                break
                        except Exception: # Other general exceptions during json.loads
                            break # Something else went wrong, stop trimming
                # If still a JSONDecodeError, or other error, re-raise as ValueError
                raise ValueError(
                    f"Could not parse model output as JSON:\n{e_fixed}\n\n---\n{text[:1500]}"
                )
            except Exception as e_fixed:  # Catch other non-JSONDecodeError exceptions too
                raise ValueError(
                    f"Could not parse model output as JSON:\n{e_fixed}\n\n---\n{text[:1500]}"
                )

    # If no '{' or '}' found, or other unhandled errors
    raise ValueError(
        f"Could not parse model output as JSON. No valid JSON object found or unexpected error.\n\n---\n{text[:1500]}"
    )


def generate_seo(transcript, language="Hinglish", niche="", current_title="", platform="YouTube Shorts"):
    prompt = build_prompt(transcript, language, niche, current_title, platform)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.06,
            pad_token_id=tokenizer.eos_token_id,
        )
    raw = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return parse_json(raw)


def render_report(d, transcript_len=0):
    """Turn the JSON into a clean, copy-paste-ready markdown report."""
    L = []
    a = L.append
    a("# 🚀 Viral SEO Package\n")
    a(f"**Niche:** {d.get('niche','—')}   |   **Transcript:** {transcript_len:,} chars\n")

    a("## 🩺 Diagnosis (ranked by impact)\n")
    for i, x in enumerate(d.get("diagnosis", []), 1):
        a(f"{i}. **[{x.get('severity','?')}]** {x.get('problem','')}\n   → *{x.get('fix','')}*\n")

    a("## 🏷️ Titles (pick one — char count shown)\n")
    for i, t in enumerate(d.get("titles", []), 1):
        a(f"{i}. `{t}`  — **{len(t)} chars**\n")

    a("## 📝 Description (copy-paste)\n")
    a("```\n" + (d.get("description") or "") + "\n```\n")

    tags = d.get("tags", [])
    a(f"## 🏷️ Tags ({len(tags)} tags, {len(', '.join(tags))} chars)\n")
    a("```\n" + ", ".join(tags) + "\n```\n")

    a("## #️⃣ Hashtags (5 max)\n")
    a(" ".join(d.get("hashtags", [])) + "\n")

    a("## 🖼️ Thumbnail prompt (paste into any AI image generator)\n")
    a("```\n" + (d.get("thumbnail_prompt") or "") + "\n```\n")
    a(f"**Thumbnail text (3 words max):** `{d.get('thumbnail_text','')}`\n")

    a("## 🪝 Hook — first 2 seconds\n")
    a((d.get("hook_advice") or "") + "\n")

    a("## 📈 Retention tips\n")
    for t in d.get("retention_tips", []):
        a(f"- {t}\n")

    a("## ⚠️ Common mistakes to avoid\n")
    for t in d.get("common_mistakes", []):
        a(f"- {t}\n")

    return "\n".join(L)


print("✅ SEO engine ready")

✅ SEO engine ready


In [7]:
# ═══════════ 🔧 PATCH — fixes JSONDecodeError "Extra data" ═══════════
import json, re, gc, torch

MAX_NEW_TOKENS = 3200   # was 2200 — it was truncating your tags list mid-word


def _strip_fences(t):
    """Remove ```json fences."""
    t = t.strip()
    if t.startswith("```"):
        t = re.sub(r"^```[A-Za-z]*\s*", "", t)
        t = re.sub(r"\s*```$", "", t)
    return t.strip()


def _repair_json(t):
    """Close unbalanced quotes/braces so TRUNCATED output still parses."""
    out, stack, in_str, esc = [], [], False, False
    for ch in t:
        if esc:
            out.append(ch); esc = False; continue
        if ch == "\\":
            out.append(ch); esc = True; continue
        if ch == '"':
            in_str = not in_str; out.append(ch); continue
        if not in_str:
            if ch == "{":
                stack.append("}")
            elif ch == "[":
                stack.append("]")
            elif ch in "}]" and stack and stack[-1] == ch:
                stack.pop()
        out.append(ch)
    s = "".join(out)
    if in_str:
        s += '"'
    s = re.sub(r",\s*$", "", s.rstrip())
    return s + "".join(reversed(stack))


def parse_json(text):
    """Pull the FIRST valid JSON value out of model output.

    Handles: prose before the JSON, ```json fences, trailing commas,
    EXTRA DATA AFTER the object (your error), and truncated output.
    """
    dec = json.JSONDecoder()
    t = _strip_fences(text)

    candidates = [t]
    for opener in ("{", "["):
        i = t.find(opener)
        if i != -1:
            candidates.append(t[i:])

    fixed = re.sub(r",\s*([}\]])", r"\1", t)   # trailing-comma repair
    candidates.append(fixed)
    if "{" in fixed:
        candidates.append(fixed[fixed.find("{"):])

    for base in (t, fixed):
        i = base.find("{")
        if i != -1:
            candidates.append(_repair_json(base[i:]))

    for cand in candidates:
        if not cand:
            continue
        try:
            obj, _ = dec.raw_decode(cand)   # ← stops at the FIRST object
            return obj
        except Exception:
            continue

    raise ValueError(f"Could not parse model output as JSON:\n---\n{text[:2000]}")


def generate_seo(transcript, language="Hinglish", niche="",
                 current_title="", platform="YouTube Shorts"):
    prompt = build_prompt(transcript, language, niche, current_title, platform)
    raw, last_err = "", None

    for attempt in range(3):
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=(TEMPERATURE if attempt == 0 else 0.4),
                do_sample=True, top_p=0.9, repetition_penalty=1.06,
                pad_token_id=tokenizer.eos_token_id,
            )
        raw = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        try:
            return parse_json(raw)
        except Exception as e:
            last_err = e
            print(f"⚠️ JSON parse attempt {attempt + 1} failed — retrying…")
            gc.collect(); torch.cuda.empty_cache()

    raise ValueError(f"Could not parse after 3 attempts: {last_err}\n---\n{raw[:2000]}")


if "LENGTH RULES" not in SYSTEM_PROMPT:
    SYSTEM_PROMPT += ('\n\nSTRICT LENGTHS: "titles" EXACTLY 5 items. '
                      '"tags" EXACTLY 25 items. "hashtags" EXACTLY 5 items. '
                      '"diagnosis" 3 to 6 items. "description" 250 to 400 words.')

print("✅ Patch applied — re-run the test cell or open the Gradio UI.")

✅ Patch applied — re-run the test cell or open the Gradio UI.


In [8]:
# Uncomment to run a quick smoke test:
#
sample = "The model context protocol MCP C# SDK has been updated to version 2.0. Analyzing did the latest 2026 728 specification reversion. This signification release introduce a stateless by default architecture allow MTP traffic to be managed by standard HTTP infrastructure and load balancer without requiring persistence sessions. A major highlight is the introductions of multi-round trip requested MRTR which enable complex interactive tools to promote for user input or additional data while remaining compatible with sessionless environments. Dispute these foundational changes. The SDK maintains backward compatibility ensure that legacy client and servers continue to function without immediate code rewrites. The update also move advanced features like apps and task into a specialized opt-in package to keep the core library lightweights. Ultimately these enhancement leverage the strength of ASP.NET core to provide a high highly scalable and modern framework for AIdriven development. So if you want to more details please visit my channel. Simply add comments what do you want to next in this topics and don't forget to lifetime."
# sample += "Na coding chahiye, na laptop. Bas mobile se sign up karo aur prompt do. "
# sample += "Maine ek shooter game banaya 60 second me, ek solar system 25 second me, "
# sample += "aur ek Minecraft jaisa game bhi. Pura free hai."

result = generate_seo(sample, language="English", niche="mcp tools")
from IPython.display import Markdown, display
display(Markdown(render_report(result, len(sample))))

print("Uncomment the block above to run a test.")


# 🚀 Viral SEO Package

**Niche:** MCP Tools   |   **Transcript:** 1,134 chars

## 🩺 Diagnosis (ranked by impact)

1. **[High]** The video focuses on technical details rather than engaging content.
   → *Create engaging visuals and simplify complex concepts for better retention.*

2. **[Medium]** No clear hook or call-to-action in the video.
   → *Add an engaging hook in the first 2 seconds and a clear call-to-action.*

3. **[Low]** Generic thumbnail text.
   → *Use specific keywords in the thumbnail text.*

## 🏷️ Titles (pick one — char count shown)

1. `MCP Tools Update: Stateless & Multi-Round Trip Requested`  — **56 chars**

2. `MCP SDK v2.0 Changes Explained`  — **30 chars**

3. `How MCP C# Updated for Modern Development`  — **41 chars**

4. `MCP Tools: Simplifying Complex Interactive Tools`  — **48 chars**

5. `What's New in MCP C# SDK 2.0?`  — **29 chars**

## 📝 Description (copy-paste)

```
Explore the latest updates to MCP C# SDK 2.0, including the introduction of stateless architecture and multi-round trip requested feature. Learn how these changes enhance scalability and compatibility for modern AI-driven development. Join the conversation and share your thoughts below!
```

## 🏷️ Tags (24 tags, 420 chars)

```
MCP Tools, MCP C# SDK, MCP 2.0, stateless architecture, multi-round trip request, modern development, AI-driven, ASP.NET core, scalability, compatibility, interactive tools, sessionless environments, backward compatibility, opt-in packages, lightweight library, software development, programming languages, API updates, software development tools, SDK, MCP C#, development frameworks, technical updates, programming tips
```

## #️⃣ Hashtags (5 max)

#MCPTools #MCP20 #MCPUpdate #SDKUpdates #TechTalk

## 🖼️ Thumbnail prompt (paste into any AI image generator)

```
A visual representation of a computer screen showing the MCP C# SDK 2.0 interface, with the focus on the update features like stateless architecture and multi-round trip request.
```

**Thumbnail text (3 words max):** `MCP C# SDK`

## 🪝 Hook — first 2 seconds

Show a quick demonstration of the new stateless architecture or multi-round trip request feature within the first 2 seconds to capture viewer attention.

## 📈 Retention tips

- Focus on visual aids to make complex concepts easier to understand.

- Use clear and concise language to explain the benefits of the update.

- Engage viewers with questions or interactive elements to maintain interest.

- Highlight the key points of the update in the first few seconds.

## ⚠️ Common mistakes to avoid

- Focusing too much on technical details without engaging the audience.

- Not providing enough context or background information.

- Neglecting to include a clear call-to-action or engagement prompt.


Uncomment the block above to run a test.


In [9]:
import gradio as gr, os, json, time, tempfile

def _from_audio(audio_path, language, niche, current_title, platform):
    if not audio_path:
        return "⚠️ Please upload an audio/video file first.", "", "", "", "", ""
    print("Transcribing:", audio_path)
    t0 = time.time()
    text = transcribe_audio(audio_path)
    print(f"Transcribed {len(text):,} chars in {time.time()-t0:.1f}s")
    return run_all(text, language, niche, current_title, platform)


def run_all(transcript, language, niche, current_title, platform):
    if not transcript or not transcript.strip():
        return "⚠️ Paste a transcript (or upload audio) first.", "", "", "", "", ""
    print("Generating SEO…")
    t0 = time.time()
    try:
        d = generate_seo(transcript.strip(), language, niche or "",
                         current_title or "", platform or "YouTube Shorts")
    except Exception as e:
        return f"❌ Generation failed: {e}", "", "", "", "", ""
    print(f"Done in {time.time()-t0:.1f}s")

    titles = "\n".join(f"{i}. {t}   [{len(t)} chars]"
                       for i, t in enumerate(d.get("titles", []), 1))
    tags   = ", ".join(d.get("tags", []))
    hash_  = " ".join(d.get("hashtags", []))
    thumb  = (d.get("thumbnail_prompt") or "") + \
             f"\n\n---\nTEXT OVERLAY (3 words max): {d.get('thumbnail_text','')}"
    report = render_report(d, len(transcript))

    # also save a downloadable copy
    os.makedirs("seo_output", exist_ok=True)
    fname = f"seo_output/seo_{int(time.time())}.md"
    with open(fname, "w", encoding="utf-8") as f:
        f.write(report)
    with open(fname.replace(".md", ".json"), "w", encoding="utf-8") as f:
        json.dump(d, f, ensure_ascii=False, indent=2)
    print("Saved:", fname)

    return report, titles, d.get("description", ""), tags, hash_, thumb


with gr.Blocks(title="🚀 Viral SEO Agent", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🚀 Viral SEO Agent — transcript / audio → complete YouTube SEO")
    gr.Markdown("Runs 100% locally on the Colab GPU. No API key.")

    with gr.Row():
        with gr.Column(scale=1):
            mode = gr.Radio(["📝 Transcript", "🎵 Audio file"], value="📝 Transcript",
                            label="Input mode")
            transcript_in = gr.Textbox(label="Paste your video transcript",
                                       lines=10, placeholder="Paste the transcript here…")
            audio_in = gr.Audio(label="…or upload audio / video file",
                                type="filepath")
            language = gr.Dropdown(["Hinglish", "Hindi", "English"],
                                   value="Hinglish", label="Language")
            niche = gr.Textbox(label="Niche / topic (optional)",
                               placeholder="e.g. free AI tool, cooking, tech tips")
            current_title = gr.Textbox(label="Current title (optional)",
                                       placeholder="If you already have one")
            platform = gr.Dropdown(["YouTube Shorts", "Instagram Reels",
                                    "YouTube Long-form", "TikTok"],
                                   value="YouTube Shorts", label="Platform")
            go = gr.Button("🚀 Generate SEO", variant="primary", size="lg")

        with gr.Column(scale=1):
            report_out  = gr.Markdown(label="Full report")
            titles_out  = gr.Textbox(label="🏷️ Titles", lines=6)
            desc_out    = gr.Textbox(label="📝 Description", lines=14)
            tags_out    = gr.Textbox(label="🏷️ Tags", lines=5)
            hash_out    = gr.Textbox(label="#️⃣ Hashtags", lines=2)
            thumb_out   = gr.Textbox(label="🖼️ Thumbnail prompt", lines=10)

    def dispatch(mode_sel, txt, aud, lang, niche_, cur, plat):
        if mode_sel == "🎵 Audio file":
            return _from_audio(aud, lang, niche_, cur, plat)
        return run_all(txt, lang, niche_, cur, plat)

    go.click(dispatch,
             inputs=[mode, transcript_in, audio_in, language, niche,
                     current_title, platform],
             outputs=[report_out, titles_out, desc_out, tags_out,
                      hash_out, thumb_out])

    gr.Markdown("---\n📁 Every run also saves `seo_output/seo_*.md` and `.json` "
                "to the Colab file browser — download them from the left sidebar.")

demo.launch(share=True, debug=False, show_error=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://84d0549a1bbd231ff1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
